<a href="https://colab.research.google.com/github/FuzzyFade/qwen35-tool-calling-sft/blob/main/Qwen35_9B_Tool_Calling_SFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen3.5-9B-Base 工具调用 SFT 微调

使用 Unsloth + TRL 对 Qwen3.5-9B-Base 进行监督微调，训练工具调用/Agent 能力。

**数据**: 100k 中英混合工具调用样本 (7 个公开数据集)

**GPU 自适应**: 自动检测 T4/L4/A100 并调整参数

---

## 运行方式
1. 点击 `运行时` → `更改运行时类型` → 选择 `T4 GPU` (免费) 或 `A100` (Pro)
2. 依次运行每个 Cell
3. 训练完成后模型自动保存到 Google Drive

## 1. 安装依赖

In [1]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install deepspeed

## 2. 检测 GPU 并设置训练参数

In [2]:
import torch

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

# 根据 GPU 自动调参
# Qwen3.5-9B 词表 248k，logits 矩阵大。用 CPU offload + grad_ckpt 省显存
if vram_gb >= 70:  # A100 80GB / H100
    CONFIG = {
        "load_in_4bit": False,
        "max_seq_length": 4096,
        "batch_size": 4,
        "grad_accum": 4,
        "lora_r": 32,
    }
    print("→ A100-80GB/H100: bf16 LoRA, batch=4, seq=4096")
elif vram_gb >= 35:  # A100 40GB
    CONFIG = {
        "load_in_4bit": False,  # bf16 精度更好
        "max_seq_length": 2048,
        "batch_size": 2,
        "grad_accum": 8,
        "lora_r": 32,
    }
    print("→ A100-40GB: bf16 LoRA r=32, batch=2, seq=2048, packing+compile")
elif vram_gb >= 20:  # L4 24GB / A10
    CONFIG = {
        "load_in_4bit": True,
        "max_seq_length": 2048,
        "batch_size": 4,
        "grad_accum": 4,
        "lora_r": 16,
    }
    print("→ L4/A10: 4-bit QLoRA, batch=4, seq=2048")
else:  # T4 16GB
    CONFIG = {
        "load_in_4bit": True,
        "max_seq_length": 1024,
        "batch_size": 2,
        "grad_accum": 8,
        "lora_r": 8,
    }
    print("→ T4: 4-bit QLoRA, batch=2, seq=1024")

print(f"\n配置: {CONFIG}")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
→ A100-40GB: bf16 LoRA r=32, batch=2, seq=2048, packing+compile

配置: {'load_in_4bit': False, 'max_seq_length': 2048, 'batch_size': 2, 'grad_accum': 8, 'lora_r': 32}


## 3. 加载模型

In [3]:
from unsloth import FastLanguageModel
import gc, torch

# 清理之前可能残留的模型显存
if 'model' in dir(): del model
if 'processor' in dir(): del processor
if 'tokenizer' in dir(): del tokenizer
gc.collect()
torch.cuda.empty_cache()

load_4bit = CONFIG["load_in_4bit"]
model, processor = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-9B-Base",
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=load_4bit,
    load_in_16bit=not load_4bit,
    dtype=None,
)

# Qwen3.5 是 VLM，Unsloth 返回 processor 而不是 tokenizer
# 需要提取内部的 tokenizer 才能用 apply_chat_template
if hasattr(processor, 'tokenizer'):
    tokenizer = processor.tokenizer
    print(f'从 processor 中提取 tokenizer: {type(tokenizer).__name__}')
else:
    tokenizer = processor

print(f"模型加载完成: {type(model).__name__}")
print(f"has chat_template: {tokenizer.chat_template is not None}")
print(f"VRAM 使用: {torch.cuda.memory_allocated()/1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/386 [00:00<?, ?B/s]

Qwen/Qwen3.5-9B-Base does not have a padding token! Will use pad_token = <|vision_pad|>.
从 processor 中提取 tokenizer: TokenizersBackend
模型加载完成: Qwen3_5ForConditionalGeneration
has chat_template: True
VRAM 使用: 18.8 GB


## 4. 配置 LoRA

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_r"],    # RSLoRA 会自动缩放，alpha=r 即可
    lora_dropout=0,                 # Unsloth 推荐 dropout=0（更快且效果相当）
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_rslora=True,                # Rank-Stabilized LoRA，高 rank 下更稳定
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"可训练参数: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

Unsloth: Making `model.base_model.model.model.language_model` require gradients
可训练参数: 58,195,968 / 9,468,009,712 (0.61%)


## 5. 准备数据

两种方式选一种:
- **方式 A**: 从本地上传已准备好的 `train.jsonl` / `valid.jsonl`
- **方式 B**: 在 Colab 中重新从 HuggingFace 下载并转换 (需要 ~5 分钟)

In [5]:
#@title 选择数据加载方式 { run: "auto" }
DATA_SOURCE = "download_fresh"  #@param ["upload_local", "download_fresh"]

import os
os.makedirs("/content/data", exist_ok=True)

In [6]:
# === 方式 A: 上传本地文件 ===
# 如果你选了 upload_local，运行这个 cell 会弹出文件选择框
if DATA_SOURCE == "upload_local":
    from google.colab import files
    print("请上传 train.jsonl 和 valid.jsonl")
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(f"/content/data/{name}", "wb") as f:
            f.write(content)
        print(f"  已保存: /content/data/{name}")
else:
    print("跳过上传，将在下一步从 HuggingFace 下载数据")

跳过上传，将在下一步从 HuggingFace 下载数据


In [7]:
# === 方式 B: 从 HuggingFace 下载并转换 ===
if DATA_SOURCE == "download_fresh" and not os.path.exists("/content/data/train.jsonl"):
    import json, hashlib, random, re
    from datasets import load_dataset

    def wrap_tool_openai(tool_def):
        if "type" in tool_def and tool_def["type"] == "function" and "function" in tool_def:
            return tool_def
        func = {"name": tool_def.get("name", "unknown"), "description": tool_def.get("description", "")}
        params = tool_def.get("parameters", {})
        if isinstance(params, dict) and "properties" in params:
            func["parameters"] = params
        else:
            func["parameters"] = {"type": "object", "properties": {}}
        return {"type": "function", "function": func}

    def make_tool_call(name, arguments):
        if isinstance(arguments, str):
            try: arguments = json.loads(arguments)
            except: arguments = {"raw": arguments}
        if not isinstance(arguments, dict): arguments = {"value": arguments}
        return {"type": "function", "function": {"name": name, "arguments": arguments}}

    def convert_sharegpt(ds, lang="en"):
        """转换 ShareGPT 格式 (glaive_zh, glaive_v2_sharegpt)"""
        results = []
        sys_msg = "你是一个有用的助手，可以调用工具来帮助用户完成任务。" if lang == "zh" else "You are a helpful assistant with access to tools."
        for row in ds:
            try:
                convs = row.get("conversations", [])
                tools_str = row.get("tools", "[]")
                tools = [wrap_tool_openai(t) for t in (json.loads(tools_str) if isinstance(tools_str, str) else tools_str) if isinstance(t, dict)] if tools_str else []
                messages = []
                if tools: messages.append({"role": "system", "content": sys_msg})
                for conv in convs:
                    rf, val = conv.get("from", ""), conv.get("value", "")
                    if rf == "human": messages.append({"role": "user", "content": val})
                    elif rf == "gpt": messages.append({"role": "assistant", "content": val})
                    elif rf == "function_call":
                        try:
                            fc = json.loads(val) if isinstance(val, str) else val
                            messages.append({"role": "assistant", "content": None, "tool_calls": [make_tool_call(fc.get("name","unknown"), fc.get("arguments",{}))]})
                        except: messages.append({"role": "assistant", "content": val})
                    elif rf == "observation":
                        tn = ""
                        for m in reversed(messages):
                            if m.get("role")=="assistant" and m.get("tool_calls"): tn=m["tool_calls"][0]["function"]["name"]; break
                        messages.append({"role": "tool", "content": val if isinstance(val,str) else json.dumps(val,ensure_ascii=False), "name": tn})
                roles = [m["role"] for m in messages]
                if "user" in roles and "assistant" in roles:
                    sample = {"messages": messages}
                    if tools: sample["tools"] = tools
                    results.append(sample)
            except: continue
        return results

    all_samples = []

    # 1. Deepexi (中文, 24.6k)
    print("[1/7] Deepexi (中文)...")
    ds = load_dataset("Deepexi/function-calling-small", split="train")
    for row in ds:
        try:
            up, ar = row.get("userPrompt",""), row.get("assistantResponse","")
            if not up or not ar: continue
            resp = json.loads(ar)
            tc = []
            if isinstance(resp, dict) and "function" in resp:
                args = resp.get("arguments",{})
                if isinstance(args,list) and args: args = args[0] if isinstance(args[0],dict) else {}
                tc = [make_tool_call(resp["function"], args)]
            msgs = [{"role":"system","content":"你是一个有用的助手，可以调用工具来帮助用户完成任务。"},{"role":"user","content":up}]
            if tc: msgs.append({"role":"assistant","content":None,"tool_calls":tc})
            else: msgs.append({"role":"assistant","content":ar})
            all_samples.append({"messages":msgs})
        except: continue
    print(f"  {len(all_samples)} samples")

    # 2. glaive_zh (中文, 1k)
    print("[2/7] glaive_zh (中文)...")
    ds = load_dataset("llamafactory/glaive_toolcall_zh", split="train")
    r = convert_sharegpt(ds, "zh")
    all_samples.extend(r)
    print(f"  {len(r)} samples")

    # 3. glaive_v2_sharegpt (英文, 100k)
    print("[3/7] glaive_v2_sharegpt (英文)...")
    ds = load_dataset("hiyouga/glaive-function-calling-v2-sharegpt", split="train")
    r = convert_sharegpt(ds, "en")
    all_samples.extend(r)
    print(f"  {len(r)} samples")

    # 4. hermes_fc (英文, 1.9k)
    print("[4/7] hermes_fc (英文)...")
    ds = load_dataset("NousResearch/hermes-function-calling-v1", split="train")
    tc_pat = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.DOTALL)
    for row in ds:
        try:
            convs = row.get("conversations",[])
            tools_str = row.get("tools","[]")
            tools = [wrap_tool_openai(t) for t in json.loads(tools_str)] if tools_str else []
            msgs = []
            if tools: msgs.append({"role":"system","content":"You are a helpful assistant with access to tools."})
            for c in convs:
                rf, val = c.get("from",""), c.get("value","")
                if rf=="system": continue
                elif rf=="human": msgs.append({"role":"user","content":val})
                elif rf=="gpt":
                    matches = tc_pat.findall(val)
                    if matches:
                        tcs = []
                        for m in matches:
                            try: d=json.loads(m); tcs.append(make_tool_call(d.get("name","?"),d.get("arguments",{})))
                            except: pass
                        if tcs: msgs.append({"role":"assistant","content":None,"tool_calls":tcs})
                        else: msgs.append({"role":"assistant","content":val})
                    else: msgs.append({"role":"assistant","content":val})
            roles=[m["role"] for m in msgs]
            if "user" in roles and "assistant" in roles:
                s={"messages":msgs}
                if tools: s["tools"]=tools
                all_samples.append(s)
        except: continue
    print(f"  done")

    # 5. ToolACE-Qwen (英文, 10.5k)
    print("[5/7] ToolACE-Qwen (英文)...")
    ds = load_dataset("tryumanshow/ToolACE-Qwen-cleaned", split="train")
    for row in ds:
        try:
            tools = [wrap_tool_openai(t) for t in json.loads(row.get("tools","[]"))] if row.get("tools") else []
            convs = json.loads(row.get("conversations","[]")) if isinstance(row.get("conversations"),str) else row.get("conversations",[])
            msgs=[]
            if tools: msgs.append({"role":"system","content":"You are a helpful assistant with access to tools."})
            for c in convs:
                role,content = c.get("role",""), c.get("content","")
                if role=="user": msgs.append({"role":"user","content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False)})
                elif role=="assistant":
                    tc_raw=c.get("tool_calls",[])
                    if tc_raw:
                        tcs=[]
                        for tc in tc_raw:
                            fd=tc.get("function",{}); args=fd.get("arguments",{})
                            if isinstance(args,str):
                                try: args=json.loads(args)
                                except: args={"raw":args}
                            tcs.append(make_tool_call(fd.get("name","?"),args))
                        msgs.append({"role":"assistant","content":None,"tool_calls":tcs})
                    else: msgs.append({"role":"assistant","content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False)})
                elif role=="tool":
                    tn=c.get("name","")
                    if not tn:
                        for m in reversed(msgs):
                            if m.get("role")=="assistant" and m.get("tool_calls"): tn=m["tool_calls"][0]["function"]["name"]; break
                    msgs.append({"role":"tool","content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False),"name":tn})
            roles=[m["role"] for m in msgs]
            if "user" in roles and "assistant" in roles:
                s={"messages":msgs}
                if tools: s["tools"]=tools
                all_samples.append(s)
        except: continue
    print(f"  done")

    # 6. Opus reasoning (英文, 2.3k)
    print("[6/7] Opus Reasoning (英文)...")
    ds = load_dataset("nohurry/Opus-4.6-Reasoning-3000x-filtered", split="train")
    for row in ds:
        p, t, s = row.get("problem",""), row.get("thinking",""), row.get("solution","")
        if p and s:
            ac = f"<think>\n{t}\n</think>\n\n{s}" if t else s
            all_samples.append({"messages":[{"role":"user","content":p},{"role":"assistant","content":ac}]})
    print(f"  done")

    # 7. OpenClaw (英文, 7.2k)
    print("[7/7] OpenClaw (英文)...")
    for split in ["train","test"]:
        try:
            ds = load_dataset("bellfire/openclaw-coder-dataset", split=split)
            for row in ds:
                raw = row.get("messages",[])
                if not raw: continue
                msgs=[]
                for m in raw:
                    role,content = m.get("role",""), m.get("content","")
                    if role in ("system","user"):
                        msgs.append({"role":role,"content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False)})
                    elif role=="assistant":
                        tc_raw=m.get("tool_calls",[])
                        if tc_raw:
                            tcs=[make_tool_call(tc.get("function",{}).get("name","?"),tc.get("function",{}).get("arguments",{})) for tc in tc_raw]
                            msgs.append({"role":"assistant","content":None,"tool_calls":tcs})
                        else: msgs.append({"role":"assistant","content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False)})
                    elif role=="tool":
                        tn=m.get("name","")
                        msgs.append({"role":"tool","content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False),"name":tn})
                roles=[m["role"] for m in msgs]
                if "user" in roles and "assistant" in roles:
                    all_samples.append({"messages":msgs})
        except: pass
    print(f"  done")

    # 去重 + 打乱 + 划分
    print(f"\n总计: {len(all_samples)} 样本")
    seen = set()
    deduped = []
    for s in all_samples:
        h = hashlib.md5(json.dumps(s["messages"],sort_keys=True,ensure_ascii=False).encode()).hexdigest()
        if h not in seen: seen.add(h); deduped.append(s)
    print(f"去重后: {len(deduped)} 样本")

    random.seed(42)
    random.shuffle(deduped)
    split_idx = int(len(deduped) * 0.9)

    for name, data in [("train.jsonl", deduped[:split_idx]), ("valid.jsonl", deduped[split_idx:])]:
        with open(f"/content/data/{name}", "w") as f:
            for s in data: f.write(json.dumps(s, ensure_ascii=False) + "\n")
        print(f"  {name}: {len(data)} 样本")

    print("数据准备完成!")
else:
    print("数据已存在，跳过")

[1/7] Deepexi (中文)...


README.md: 0.00B [00:00, ?B/s]

function-calling-small_aliyun_openapi_V2(…):   0%|          | 0.00/112M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24608 [00:00<?, ? examples/s]

  24192 samples
[2/7] glaive_zh (中文)...


...

Generating train split: 0 examples [00:00, ? examples/s]

Failed to load JSON from file '/root/.cache/huggingface/hub/datasets--bellfire--openclaw-coder-dataset/snapshots/ca551f730d82fc43666010f9ba760c1ed0fd88b6/data/train.jsonl' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: Column(/messages/[]/content) changed from string to object in row 0
ERROR:datasets.packaged_modules.json.json:Failed to load JSON from file '/root/.cache/huggingface/hub/datasets--bellfire--openclaw-coder-dataset/snapshots/ca551f730d82fc43666010f9ba760c1ed0fd88b6/data/train.jsonl' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: Column(/messages/[]/content) changed from string to object in row 0


Generating train split: 0 examples [00:00, ? examples/s]

Failed to load JSON from file '/root/.cache/huggingface/hub/datasets--bellfire--openclaw-coder-dataset/snapshots/ca551f730d82fc43666010f9ba760c1ed0fd88b6/data/train.jsonl' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: Column(/messages/[]/content) changed from string to object in row 0
ERROR:datasets.packaged_modules.json.json:Failed to load JSON from file '/root/.cache/huggingface/hub/datasets--bellfire--openclaw-coder-dataset/snapshots/ca551f730d82fc43666010f9ba760c1ed0fd88b6/data/train.jsonl' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: Column(/messages/[]/content) changed from string to object in row 0


  done

总计: 140503 样本
去重后: 99907 样本
  train.jsonl: 89916 样本
  valid.jsonl: 9991 样本
数据准备完成!


In [8]:
# 验证数据格式
import json

with open("/content/data/train.jsonl") as f:
    sample = json.loads(f.readline())

print(f"样本字段: {list(sample.keys())}")
print(f"消息轮次: {len(sample['messages'])}")
print(f"有 tools: {'tools' in sample}")

# 测试 apply_chat_template
try:
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tools=sample.get("tools"),
        tokenize=False,
    )
    print(f"\nTokenized 长度: {len(tokenizer.encode(text))} tokens")
    print(f"前 300 字符:\n{text[:300]}")
except Exception as e:
    print(f"ERROR: {e}")

样本字段: ['messages', 'tools']
消息轮次: 7
有 tools: True

Tokenized 长度: 472 tokens
前 300 字符:
<|im_start|>system
# Tools

You have access to the following functions:

<tools>
{"type": "function", "function": {"name": "generate_qr_code", "description": "Generate a QR code for a given text", "parameters": {"type": "object", "properties": {"text": {"type": "string", "description": "The text to 


## 6. 格式化数据集

In [9]:
import json as _json
from datasets import Dataset

def load_and_format(filepath, tokenizer):
    """直接读 JSONL 并转为纯文本，绕过 datasets 的类型推断问题"""
    texts = []
    skipped = 0
    errors = []
    with open(filepath) as f:
        for i, line in enumerate(f):
            try:
                sample = _json.loads(line)
                messages = sample.get("messages", [])
                tools = sample.get("tools") or None
                # 清洗 messages，确保类型正确
                cleaned = []
                for m in messages:
                    msg = {"role": m["role"], "content": m.get("content")}
                    if m.get("tool_calls"):
                        tcs = []
                        for tc in m["tool_calls"]:
                            func = tc.get("function", tc)
                            args = func.get("arguments", {})
                            if isinstance(args, str):
                                try: args = _json.loads(args)
                                except: args = {}
                            if not isinstance(args, dict): args = {}
                            tcs.append({"type": "function", "function": {"name": func.get("name", ""), "arguments": args}})
                        msg["tool_calls"] = tcs
                        msg["content"] = None
                    if m.get("name"):
                        msg["name"] = m["name"]
                    cleaned.append(msg)
                text = tokenizer.apply_chat_template(
                    cleaned, tools=tools, tokenize=False, add_generation_prompt=False,
                )
                if text:
                    texts.append(text)
            except Exception as e:
                skipped += 1
                if len(errors) < 5:
                    errors.append(f"  Row {i}: {type(e).__name__}: {e}")
    print(f"  {filepath}: {len(texts)} OK, {skipped} skipped")
    if errors:
        print("  First errors:")
        for err in errors: print(err)
    return Dataset.from_dict({"text": texts})

print("格式化训练集...")
train_ds = load_and_format("/content/data/train.jsonl", tokenizer)
print("格式化验证集...")
valid_ds = load_and_format("/content/data/valid.jsonl", tokenizer)

print(f"\n训练集: {len(train_ds)} 样本")
print(f"验证集: {len(valid_ds)} 样本")

格式化训练集...
  /content/data/train.jsonl: 89916 OK, 0 skipped
格式化验证集...
  /content/data/valid.jsonl: 9991 OK, 0 skipped

训练集: 89916 样本
验证集: 9991 样本


## 7. 开始训练

In [10]:
from trl import SFTTrainer, SFTConfig
import torch.fx.experimental._config as fx_config
fx_config.meta_nonzero_assume_all_nonzero = True

from unsloth import FastLanguageModel
FastLanguageModel.for_training(model)

training_args = SFTConfig(
    output_dir="/content/output",
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    max_steps=1000,
    learning_rate=2e-5,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    bf16=vram_gb >= 20,
    fp16=vram_gb < 20,
    logging_steps=10,
    save_steps=1000,
    save_total_limit=1,
    seed=42,
    optim="adamw_8bit",
    max_seq_length=CONFIG["max_seq_length"],
    dataset_text_field="text",
    packing=True,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    torch_compile=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    train_dataset=train_ds,
    args=training_args,
)

print(f"有效 batch size: {CONFIG['batch_size'] * CONFIG['grad_accum']}")
print(f"总训练步数: 1000")
print("\n开始训练...\n")

trainer.train()

Unsloth: Sample packing skipped (processor-based model detected).


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/89916 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248044}.


有效 batch size: 16
总训练步数: 1000

开始训练...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 89,916 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 58,195,968 of 9,468,009,712 (0.61% trained)


Unsloth: Enabled auto compiling
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,5.481746
20,5.236597
30,4.597068
40,3.872347
50,3.540923
60,3.652761
70,3.768167
80,3.450118
90,3.646283
100,3.475571


TrainOutput(global_step=1000, training_loss=3.1379985656738283, metrics={'train_runtime': 19595.3754, 'train_samples_per_second': 0.817, 'train_steps_per_second': 0.051, 'total_flos': 6.716172857884362e+17, 'train_loss': 3.1379985656738283, 'epoch': 0.17794385871257618})

## 8. 保存模型

In [11]:
# 保存 LoRA 适配器
model.save_pretrained("/content/output/lora_adapter")
processor.save_pretrained("/content/output/lora_adapter")
print("LoRA 适配器已保存")

# 保存合并后的完整模型
model.save_pretrained_merged("/content/output/merged_bf16", processor, save_method="merged_16bit")
print("合并模型 (bf16) 已保存")

LoRA 适配器已保存
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 4 files from cache to `/content/output/merged_bf16`:   0%|          | 0/4 [00:00<?, ?it/s]
Unsloth: Copying 4 files from cache to `/content/output/merged_bf16`:  25%|██▌       | 1/4 [00:13<00:39, 13.15s/it]
Unsloth: Copying 4 files from cache to `/content/output/merged_bf16`:  50%|█████     | 2/4 [00:25<00:25, 12.92s/it]
Unsloth: Copying 4 files from cache to `/content/output/merged_bf16`:  75%|███████▌  | 3/4 [00:40<00:13, 13.64s/it]
Unsloth: Copying 4 files from cache to `/content/output/merged_bf16`: 100%|██████████| 4/4 [00:49<00:00, 12.29s/it]


Successfully copied all 4 files from cache to `/content/output/merged_bf16`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 30783.88it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:04<00:00, 16.03s/it]


Unsloth: Merge process complete. Saved to `/content/output/merged_bf16`
合并模型 (bf16) 已保存


## 9. 测试工具调用

In [16]:
FastLanguageModel.for_inference(model)

test_cases = [
    {
        "name": "中文-天气查询",
        "messages": [{"role": "user", "content": "请帮我查询北京今天的天气"}],
        "tools": [{"type": "function", "function": {
            "name": "get_weather", "description": "查询指定城市天气",
            "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
        }}],
    },
    {
        "name": "EN-File Search",
        "messages": [{"role": "user", "content": "Search for all Python files in the src directory"}],
        "tools": [{"type": "function", "function": {
            "name": "search_files", "description": "Search files by pattern",
            "parameters": {"type": "object", "properties": {"dir": {"type": "string"}, "pattern": {"type": "string"}}, "required": ["dir", "pattern"]}
        }}],
    },
    {
        "name": "中文-普通对话",
        "messages": [{"role": "user", "content": "你好，请介绍一下你自己"}],
        "tools": None,
    },
]

for tc in test_cases:
    print(f"\n{'='*50}")
    print(f"测试: {tc['name']}")
    print(f"User: {tc['messages'][0]['content']}")
    if tc.get('tools'): print(f"Tools: {[t['function']['name'] for t in tc['tools']]}")

    input_text = tokenizer.apply_chat_template(
        tc["messages"], tools=tc["tools"], tokenize=False, add_generation_prompt=True
    )
    # 显式使用 text=input_text 避免被误认为图像
    inputs = processor(text=input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, do_sample=True)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    if "<|im_end|>" in resp: resp = resp[:resp.index("<|im_end|>")]
    print(f"\nAssistant: {resp.strip()}")



测试: 中文-天气查询
User: 请帮我查询北京今天的天气
Tools: ['get_weather']

Assistant: 用户要求查询北京今天的天气。我需要使用get_weather函数，并提供城市参数"北京"。
</think>

<tool_call>
<function=get_weather>
<parameter=city>
北京
</parameter>
</function>
</tool_call>

测试: EN-File Search
User: Search for all Python files in the src directory
Tools: ['search_files']

Assistant: The user wants to search for all Python files in the src directory. I need to use the search_files function with:
- dir: "src" 
- pattern: "*.py" (to match all Python files)

Let me make this function call.
</think>

<tool_call>
<function=search_files>
<parameter=dir>
src
</parameter>
<parameter=pattern>
*.py
</parameter>
</function>
</tool_call>

测试: 中文-普通对话
User: 你好，请介绍一下你自己

Assistant: 嗯，用户让我介绍一下自己，这是个很常见的开场问题。需要清晰简洁地说明我的核心功能，同时保持友好。

想到了可以从几个方面展开：基本定位、主要能力、使用方式、服务特点。这样结构会比较清晰。

需要注意避免过于技术化的描述，用用户能理解的语言。可以强调我的实用性和免费特性，这是用户可能关心的点。

最后可以加个开放性问题，引导用户进一步互动。这样既完成了自我介绍，又促进了对话继续。
</think>

你好！我是由百度智能云开发的人工智能助手，主要功能是提供信息查询、内容生成、语言处理等服务。我可以帮你解答问题、进行多语言翻译、总结文本、生成创意内容，甚至协助

## 10. 导出 GGUF (可选)

导出后可在本地用 Ollama / LM Studio 运行

In [17]:
# 导出 GGUF Q4_K_M
model.save_pretrained_gguf(
    "/content/output/gguf",
    processor,
    quantization_method="q4_k_m",
)
print("GGUF 已导出!")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 4 files from cache to `/content/output/gguf`:   0%|          | 0/4 [00:00<?, ?it/s]
Unsloth: Copying 4 files from cache to `/content/output/gguf`:  25%|██▌       | 1/4 [00:14<00:42, 14.20s/it]
Unsloth: Copying 4 files from cache to `/content/output/gguf`:  50%|█████     | 2/4 [00:28<00:28, 14.20s/it]
Unsloth: Copying 4 files from cache to `/content/output/gguf`:  75%|███████▌  | 3/4 [00:41<00:13, 13.90s/it]
Unsloth: Copying 4 files from cache to `/content/output/gguf`: 100%|██████████| 4/4 [00:50<00:00, 12.72s/it]


Successfully copied all 4 files from cache to `/content/output/gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 32704.12it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:33<00:00, 23.35s/it]


Unsloth: Merge process complete. Saved to `/content/output/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/output/gguf_gguf/Qwen3.5-9B-Base.BF16.gguf', '/content/output/gguf_gguf/Qwen3.5-9B-Base.BF16-mmproj.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/output/gguf_gguf/Qwen3.5-9B-Base.Q4_K_M.gguf', '/content/output/gguf_gguf/Qwen3.5-9B-Base.BF16-mmproj.gguf']
Unsloth: No Ollama template mapping found for model 'Qwen/Qwen3.5-9B-Base'. Skipping Ollama Modelfile


Unsloth: example usage for Multimodal LLMs: /root/.unsloth/llama.cpp/llama-mtmd-cli -m /content/output/gguf_gguf/Qwen3.5-9B-Base.Q4_K_M.gguf --mmproj /content/output/gguf_gguf/Qwen3.5-9B-Base.BF16-mmproj.gguf
Unsloth: load image inside llama.cpp runner: /image test_image.jpg
Unsloth: Prompt model to describe the image
GGUF 已导出!


## 11. 保存到 Google Drive (防止 Colab 断连丢失)

In [19]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

!mkdir -p /content/drive/MyDrive/qwen35-tool-calling
!cp -r /content/output/lora_adapter /content/drive/MyDrive/qwen35-tool-calling/
!cp -r /content/output/gguf /content/drive/MyDrive/qwen35-tool-calling/ 2>/dev/null || true

print("模型已保存到 Google Drive: /MyDrive/qwen35-tool-calling/")

Mounted at /content/drive
模型已保存到 Google Drive: /MyDrive/qwen35-tool-calling/


## 12. 推送到 HuggingFace Hub (可选)

In [14]:
HF_TOKEN = "hf_xxxxxx"  # 取消注释并填入你的 token
HF_REPO = "icecee/Qwen3.5-9B-Tool-Calling"  # 你的 repo

model.push_to_hub_merged(HF_REPO, processor, save_method="merged_16bit", token=HF_TOKEN)
print(f"已推送到: https://huggingface.co/{HF_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ol-Calling/tokenizer.json:  40%|####      | 8.00MB / 20.0MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   5%|4         |  160MB / 3.33GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:02<00:00, 75.54s/it]


Unsloth: Merge process complete. Saved to `/content/icecee/Qwen3.5-9B-Tool-Calling`
已推送到: https://huggingface.co/icecee/Qwen3.5-9B-Tool-Calling
